# Step 1: Installing required library
    - We will need psycopg2-binary for connection with Postgres SQL and running queries 
    - Pandas for dataframe ( stroring data for data manipuitaion, cealning, calcutions etc.)
    - sqlalchemy for creating database connections and interact with databases
    - plotly, seaborn,, matplotlib for visualizations

In [1]:
pip install psycopg2-binary pandas sqlalchemy plotly seaborn matplotlib

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 2.8/2.8 MB 33.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


# Step 2 - Connecting with DB
 - Connecting with Postgres.
 - I have created a fallback logic to load a csv if db connection fails or .env doesnt exists
 - Fallback logic is there to ensure any 3rd person using this notebook and still execute each cell.
 - I have uploaded a clean data that can be used.

In [2]:
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# Load .env if present
load_dotenv(".env", override=True)

DATABASE_URL = os.getenv("DB_URL")

if DATABASE_URL:
    try:
        engine = create_engine(DATABASE_URL)

        with engine.connect() as conn:
            result = conn.execute(text("SELECT COUNT(*) FROM candidate_log"))
            print("Connected! Total rows:", result.fetchone()[0])

        df = pd.read_sql("SELECT * FROM candidate_log", engine)
        print("Loaded data from database")

    except Exception as e:
        print(f"Database connection failed: {e}")

        csv_path = Path.cwd() / "data" / "cleaned_data.csv"
        df = pd.read_csv(csv_path)
        print("Loaded data from CSV fallback")

else:
    csv_path = Path.cwd() / "data" / "cleaned_data.csv"
    df_filtered = pd.read_csv(csv_path)
    print("Loaded data from CSV fallback")

Connected! Total rows: 8246901
Loaded data from database


# Note - If loading data from csv file, please skip step 3.B, 4.1, 4.2 and 4.2.1, as filtered data ( removing auto save event) is already store in form of CSV file.

# Step 3.A - Knowing our Data
 - We check numbers of columns there data type
 - We check a sample of data for better understanding the data at hand.

In [3]:
df_sample = df.head()
print(df_sample.shape)
print(df_sample.dtypes)
df_sample.head()

(5, 12)
log_id                          int64
candidate_id                   object
logged_at              datetime64[ns]
subject_id                     object
candidate_status               object
question_display_id            object
activity                       object
question_response              object
all_options                    object
question_language              object
question_section               object
question_type                  object
dtype: object


,log_id,candidate_id,logged_at,subject_id,candidate_status,question_display_id,activity,question_response,all_options,question_language,question_section,question_type
0,471580,OTAwMjAwNTY5NQ==,2025-09-23 09:33:23.693,1,Section 1 Question 24,24,Auto Save,A,"A,B,C,D",EN,1,MCQ
1,471581,MTQwMjAxOTk0NA==,2025-09-23 09:21:57.480,1,Section 4 Question 19,19,Auto Save,D,"A,B,C,D",EN,4,MCQ
2,471581,MTYwMTA1Mzk4OA==,2025-09-23 09:31:42.131,1,Section 4 Question 25,25,UnMark for Review & Next,None,"A,B,C,D",EN,4,MCQ
3,471581,MjQwNDAwODI4Mw==,2025-09-23 12:50:30.795,1,Section 1 Question 24,24,Auto Save,B,"A,B,C,D",HI,1,MCQ
4,471581,OTAwMTAyNDM4MQ==,2025-09-23 09:37:37.651,1,Section 3 Question 5,5,Auto Save,D,"A,B,C,D",EN,3,MCQ


# Step 3.B - Understanding the data.
 -  We already know from the problem statement and above sample data that most of data is system generated data.
 -  System generated data are not use for us, we check its percentage and then remove it.
 -  Section 3.B can only be executed if you are connection with DB and not CSV.

In [4]:
query = """
SELECT 
    activity,
    COUNT(*) as count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct
FROM candidate_log
GROUP BY activity
ORDER BY count DESC
"""
df_activity = pd.read_sql(query, engine)
print(df_activity)

                   activity    count    pct
0                 Auto Save  7482437  90.73
1    Mark for Review & Next   536510   6.51
2  UnMark for Review & Next   227954   2.76


# Step 4 - Data Cleaning & Transformation

### Objective
Prepare raw `candidate_log` data for behavioural analysis by:
- Establishing the correct composite primary key
- Filtering out system-generated Auto Save noise
- Fixing data types for numeric columns
- Standardising missing values in `question_response`
- Engineering new features useful for analysis (hour, time slots etc.)

## Step 4.1: Composite Primary Key

The schema explicitly states that `log_id` is unique ONLY when combined 
with `candidate_id`. We create a composite key to correctly identify each row.

In [5]:
df['composite_key'] = df['log_id'].astype(str) + "_" + df['candidate_id'].astype(str)

total_rows = len(df)
unique_keys = df['composite_key'].nunique()

print(f"Total rows       : {total_rows:,}")
print(f"Unique composite keys: {unique_keys:,}")
print(f"Duplicates       : {total_rows - unique_keys:,}")

Total rows       : 8,246,901
Unique composite keys: 8,246,901
Duplicates       : 0


## Step 4.2: Filter Out Auto Save Events

90.73% of rows are system-generated Auto Save pings.
These are NOT human interactions and would severely distort 
any behavioural analysis if retained.

We keep only:
- `Mark for Review & Next` → candidate flagged question and moved on
- `UnMark for Review & Next` → candidate removed the flag

In [6]:
df_raw_count = len(df)

df_filtered = df[df['activity'] != 'Auto Save'].copy()

print(f"Before filtering : {df_raw_count:,} rows")
print(f"After filtering  : {len(df_filtered):,} rows")
print(f"Removed          : {df_raw_count - len(df_filtered):,} Auto Save rows")
print(f"\nRemaining activities:")
print(df_filtered['activity'].value_counts())

Before filtering : 8,246,901 rows
After filtering  : 764,464 rows
Removed          : 7,482,437 Auto Save rows

Remaining activities:
activity
Mark for Review & Next      536510
UnMark for Review & Next    227954
Name: count, dtype: int64


### Step 4.2.1: Saving the cleaned data.

In [8]:
os.makedirs("data", exist_ok=True)
df_filtered.to_csv("data/cleaned_data.csv", index=False)
print(f"Saved cleaned data → data/cleaned_data.csv ({len(df_filtered):,} rows)")

Saved cleaned data → data/cleaned_data.csv (764,464 rows)


## Step 4.3: Fix Data Types

Several columns were loaded as 'object' (string) but represent 
numeric values. We cast them to integers for correct aggregation and sorting.

In [11]:
df_filtered['question_section'] = pd.to_numeric(df_filtered['question_section'], errors='coerce').astype('Int64')
df_filtered['question_display_id'] = pd.to_numeric(df_filtered['question_display_id'], errors='coerce').astype('Int64')
df_filtered['subject_id'] = pd.to_numeric(df_filtered['subject_id'], errors='coerce').astype('Int64')
df_filtered['logged_at'] = pd.to_datetime(df_filtered['logged_at'])

print("Updated dtypes:")
print(df_filtered[['question_section','question_display_id','subject_id','logged_at']].dtypes)

Updated dtypes:
question_section                Int64
question_display_id             Int64
subject_id                      Int64
logged_at              datetime64[ns]
dtype: object


## Step 4.4: Standardise Missing Values in `question_response`

Unanswered questions appear as `None` in the database.
We replace these with `"No Response"` to make them 
explicit and easy to filter/group in analysis.

In [12]:
df_filtered['question_response'] = df_filtered['question_response'].fillna('No Response')

print("Response value counts:")
print(df_filtered['question_response'].value_counts())

Response value counts:
question_response
No Response    764464
Name: count, dtype: int64


## Step 4.5: Feature Engineering

We extract time-based features from `logged_at` to enable 
temporal analysis — peak activity hours, exam pacing etc.

In [13]:
df_filtered['hour'] = df_filtered['logged_at'].dt.hour
df_filtered['minute'] = df_filtered['logged_at'].dt.minute

df_filtered['time_slot'] = pd.cut(
    df_filtered['hour'],
    bins=[8, 10, 12, 14, 16, 18, 20],
    labels=['9-10 AM', '11 AM-12 PM', '1-2 PM', '3-4 PM', '5-6 PM', '7 PM+']
)

print("Feature engineering done")
print(df_filtered[['logged_at', 'hour', 'time_slot']].head())

Feature engineering done
                  logged_at  hour time_slot
2   2025-09-23 09:31:42.131     9   9-10 AM
56  2025-09-23 09:06:34.716     9   9-10 AM
70  2025-09-23 09:51:25.549     9   9-10 AM
102 2025-09-23 09:31:35.146     9   9-10 AM
134 2025-09-23 16:47:02.258    16    3-4 PM


## Step 4.6: Final Cleaned Dataset Summary

In [14]:
print("=" * 45)
print("      CLEANED DATASET SUMMARY")
print("=" * 45)
print(f"Total rows         : {len(df_filtered):,}")
print(f"Unique candidates  : {df_filtered['candidate_id'].nunique():,}")
print(f"Sections covered   : {sorted(df_filtered['question_section'].dropna().unique().tolist())}")
print(f"Activities         : {df_filtered['activity'].unique().tolist()}")
print(f"Date range         : {df_filtered['logged_at'].min()} → {df_filtered['logged_at'].max()}")
print(f"Languages          : {df_filtered['question_language'].unique().tolist()}")
print("=" * 45)

      CLEANED DATASET SUMMARY
Total rows         : 764,464
Unique candidates  : 79,407
Sections covered   : [1, 2, 3, 4]
Activities         : ['UnMark for Review & Next', 'Mark for Review & Next']
Date range         : 2025-09-23 09:00:13.210000 → 2025-09-23 19:12:32.734000
Languages          : ['EN', 'HI']
